In [1]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth.chat_templates import train_on_responses_only, get_chat_template
from unsloth import FastModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-10-01 07:42:59.744085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759304580.079587      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759304580.180086      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HUGGING_FACE_TOKEN")
    upload_hf_token = user_secrets.get_secret("UPLOAD_HF_TOKEN")
except Exception as e:
    print("Could not retrieve Hugging Face token. Please ensure it is stored as a Kaggle secret named 'HUGGING_FACE_TOKEN'.")
    hf_token = None
    upload_hf_token = None

In [4]:
from datasets import load_dataset, Dataset
import pandas as pd

dataset_id = "tmnam20/ViMedAQA"

def create_stratified_subset(
    dataset: Dataset, 
    stratify_on_col: str, 
    subset_percentage: float = None, 
    num_samples: int = None,
    num_bins: int = 5, 
    seed: int = 42
) -> Dataset:
    """
    Creates a stratified random subset of a Hugging Face Dataset, 
    based on either a percentage or a fixed number of samples.

    Args:
        dataset: The input Hugging Face Dataset.
        stratify_on_col: The name of the text column for stratification.
        subset_percentage: The desired size of the subset (e.g., 0.10 for 10%).
        num_samples: The desired absolute number of samples for the subset.
        num_bins: The number of bins to create for stratification.
        seed: The random seed for reproducibility.
        
    **Note:** You must specify EITHER `subset_percentage` OR `num_samples`, but not both.

    Returns:
        A new, smaller, stratified Hugging Face Dataset.
    """
    # --- Input Validation ---
    if (subset_percentage is None and num_samples is None) or \
       (subset_percentage is not None and num_samples is not None):
        raise ValueError("You must specify EITHER `subset_percentage` OR `num_samples`, but not both.")

    df = dataset.to_pandas()

    # --- Determine Total Samples Needed ---
    if subset_percentage is not None:
        if not (0 < subset_percentage <= 1.0):
            raise ValueError("subset_percentage must be between 0 and 1.")
        total_samples_needed = int(len(df) * subset_percentage)
        print(f"Stratifying to create a {subset_percentage*100:.1f}% subset ({total_samples_needed} samples)...")
    else: # num_samples is not None
        if not (0 < num_samples <= len(df)):
            raise ValueError(f"num_samples must be between 1 and the dataset size of {len(df)}.")
        total_samples_needed = num_samples
        print(f"Stratifying to create a subset of {num_samples} samples...")
    
    # --- Stratification Logic (mostly unchanged) ---
    df['length_col'] = df[stratify_on_col].str.len().fillna(0)
    df['length_bin'] = pd.qcut(df['length_col'], q=num_bins, labels=False, duplicates='drop')

    actual_num_bins = df['length_bin'].nunique()
    samples_per_bin = max(1, total_samples_needed // actual_num_bins)

    print(f"  > Sampling {samples_per_bin} from each of the {actual_num_bins} bins...")

    subset_df = df.groupby('length_bin', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), samples_per_bin), random_state=seed), # Use min() to avoid errors on small groups
        include_groups=False
    )
    
    # Clean up temporary columns
    subset_df = subset_df.drop(columns=['length_col'])

    return Dataset.from_pandas(subset_df)

# --- Usage ---
# 1. Load the full datasets
full_train_dataset = load_dataset(dataset_id, split="train")
full_eval_dataset = load_dataset(dataset_id, split="validation")

FULL_FINETUNE = True
if FULL_FINETUNE:
    dataset = full_train_dataset
    eval_subset = full_eval_dataset
else:
    # 2. Use your function to create the subsets
    dataset = create_stratified_subset(
        dataset=full_train_dataset,
        # subset_percentage=0.10, # 10%
        stratify_on_col='context', # Stratify based on context length
        num_samples=1
    )
    
    eval_subset = create_stratified_subset(
        dataset=full_eval_dataset,
        # subset_percentage=0.20, # 20%
        stratify_on_col='context',
        num_samples=1
    )

print(f"\nFinal training subset size: {len(dataset)}")
print(f"Final evaluation subset size: {len(eval_subset)}")

README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/20.6M [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/1.21M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39881 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2217 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2215 [00:00<?, ? examples/s]


Final training subset size: 39881
Final evaluation subset size: 2215


In [5]:
def formatting_func(examples):
    texts = []
    # Get the number of examples in the batch
    num_examples = len(examples['question'])
    
    for i in range(num_examples):
        # Format a single example from the batch
        system_prompt = (
            "You are a medical expert AI. Based on your expertise, answer the following Question in Vietnamese, using ONLY the provided Context below:\n\n."
            f"### Context:\n{examples['context'][i]}"
        )
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": examples["question"][i]},
            {"role": "assistant", "content": examples["answer"][i]}
        ]
        
        # Apply the chat template to this single example
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text.removeprefix('<bos>'))
        
    return texts

<a name="Train"></a>
### Train the model
Now let's train our model. We do 100 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [6]:
from huggingface_hub import snapshot_download

print("Starting safe pre-download to cache the model files...")

# This command downloads all necessary files for the model to the local
# Hugging Face cache. It is a file operation and does NOT load the model
# into memory or onto the GPU.
snapshot_download(
    repo_id="arcee-ai/Arcee-VyLinh",
    token=hf_token,
    # Make sure to set `allow_patterns` if you only need specific files
    # but for a full model, this is sufficient.
)

print("✅ Model files have been successfully cached.")

Starting safe pre-download to cache the model files...


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

mergekit_config.yml:   0%|          | 0.00/156 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/3.70G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

✅ Model files have been successfully cached.


In [7]:
# def main():
from trl import SFTTrainer, SFTConfig
import torch

max_seq_length = 512 # For ViMedAQA

model, tokenizer = FastModel.from_pretrained(
    model_name = "arcee-ai/Arcee-VyLinh",
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    token = hf_token,
)

R_VAL = 32

model = FastModel.get_peft_model(
    model,
    r = R_VAL, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = R_VAL * 2,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    # init_lora_weights = "loftq",
    loftq_config = None, # And LoftQ, {"loftq_iter": 1}
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen2.5", # For arcee-vylinh
)

==((====))==  Unsloth 2025.9.10: Fast Qwen2 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth: Making `model.base_model.model.model` require gradients


In [8]:
# Replace your old "formatting_and_tokenizing_func" with this one.
def create_prompt_and_labels(examples):
    
    all_input_ids = []
    all_labels = []
    
    # Define the instruction and response markers
    instruction_part = "<|im_start|>user\n"
    response_part = "<|im_start|>assistant\n"
    
    num_examples = len(examples['question'])
    for i in range(num_examples):
        # 1. Format the full text prompt
        system_prompt = (
            "You are a medical expert AI. Based on your expertise, answer the following Question in Vietnamese, using ONLY the provided Context below:\\n\\n."
            f"### Context:\\n{examples['context'][i]}"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": examples["question"][i]},
        ]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        # 2. Add the assistant's answer to create the full text
        full_text = prompt_text + examples["answer"][i] + tokenizer.eos_token
        
        # 3. Tokenize the full text
        tokenized_full = tokenizer(full_text, truncation=True, padding=False, max_length=max_seq_length, add_special_tokens=False)
        
        # 4. Tokenize just the prompt part to find its length
        tokenized_prompt = tokenizer(prompt_text, truncation=True, padding=False, max_length=max_seq_length, add_special_tokens=False)
        
        prompt_length = len(tokenized_prompt['input_ids'])
        
        # 5. Create labels: copy input_ids, then mask the prompt part
        labels = list(tokenized_full['input_ids'])
        labels[:prompt_length] = [-100] * prompt_length
        
        all_input_ids.append(tokenized_full['input_ids'])
        all_labels.append(labels)
        
    return {
        "input_ids": all_input_ids,
        "labels": all_labels,
        "attention_mask": [ [1] * len(ids) for ids in all_input_ids ], # Create attention mask
    }

# --- This is the one-time operation that does all the CPU work upfront ---
print("Starting one-time pre-processing and manual label creation...")
# Important: remove old columns to avoid conflicts
tokenized_dataset = dataset.map(
    create_prompt_and_labels,
    batched=True,
    num_proc=os.cpu_count() // 2,
    remove_columns=dataset.column_names,
)
tokenized_eval_subset = eval_subset.map(
    create_prompt_and_labels,
    batched=True,
    num_proc=os.cpu_count() // 2,
    remove_columns=eval_subset.column_names,
)
print("✅ Datasets have been pre-processed with manual labels.")

Starting one-time pre-processing and manual label creation...


Map (num_proc=2):   0%|          | 0/39881 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/2215 [00:00<?, ? examples/s]

✅ Datasets have been pre-processed with manual labels.


In [9]:
# # Run this code right after the .map() call finishes
# print("Verifying the first sample of the tokenized dataset...")

# sample = tokenized_dataset[0]

# # --- Check the input_ids (what the model sees) ---
# print("\n--- Full Input (what model sees) ---")
# print(tokenizer.decode(sample['input_ids']))

# # --- Check the labels (what the model learns from) ---
# print("\n--- Labels (what model learns from) ---")
# # Create a copy of the labels, replacing -100 with a pad token for decoding
# decoded_labels = [label if label != -100 else tokenizer.pad_token_id for label in sample['labels']]
# print(tokenizer.decode(decoded_labels, skip_special_tokens=True))

# # --- Direct comparison of IDs ---
# print("\n--- Raw IDs Comparison ---")
# print(f"Input IDs:\n{sample['input_ids']}")
# print(f"Labels:\n{sample['labels']}")

In [10]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset=tokenized_dataset,      # Use the new tokenized dataset
    eval_dataset=tokenized_eval_subset,   # Use the new tokenized eval dataset
    formatting_func=None,                 # IMPORTANT: Turn this off
    dataset_text_field="input_ids",       # Tell the trainer to use the tokenized inputs directly
    args = SFTConfig(
        # dataset_text_field = "text",
        per_device_train_batch_size = 8, # 8
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5, # 5
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 100, # 100
        learning_rate = 2e-5,
        # 5e-5
        # Reduce to 2e-5 for long training runs
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir="outputs",
        report_to = "none", # Use this for WandB etc
        eval_strategy = "steps",
        eval_steps = 250,
        save_strategy = "steps",
        save_steps = 250,
        save_total_limit = 5,

        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
    ),
)

# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

print("Starting training with fully pre-processed data...")
trainer_stats = trainer.train()

# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

GPU = Tesla T4. Max memory = 14.741 GB.
2.953 GB of memory reserved.
Starting training with fully pre-processed data...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 39,881 | Num Epochs = 1 | Total steps = 1,247
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
250,0.401900,0.410737
500,0.454400,0.403630
750,0.403200,0.398508
1000,0.332900,0.394546


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


23224.3115 seconds used for training.
387.07 minutes used for training.
Peak reserved memory = 5.436 GB.
Peak reserved memory for training = 2.483 GB.
Peak reserved memory % of max memory = 36.877 %.
Peak reserved memory for training % of max memory = 16.844 %.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [11]:
# In the last cell:
model_name = "arcee-vylinh-vimedaqa-qlora-adapters"

try:
    print(f"\nAttempting to push adapters to Hugging Face Hub: haitransistor/{model_name}")
    # Online saving (best effort)
    model.push_to_hub("haitransistor/" + model_name, token=upload_hf_token) 
    tokenizer.push_to_hub("haitransistor/" + model_name, token=upload_hf_token)
    print("✅ Successfully pushed to Hub.")
except Exception as e:
    print(f"⚠️ WARNING: Failed to push to Hugging Face Hub. Error: {e}")
    # Local saving (fallback)
    print("Attempting local save as fallback...")

    import os

    # Define the base output directory for Kaggle
    kaggle_output_dir = "/kaggle/working/"
    
    # Create a specific path for your final model within the output directory
    # os.path.join is a robust way to create file paths.
    local_path = os.path.join(kaggle_output_dir, f"{model_name}")
    
    print(f"Saving final model and tokenizer to: {local_path}")
    
    # Save the model and tokenizer to the specified path
    model.save_pretrained(local_path)
    tokenizer.save_pretrained(local_path)
    
    print(f"✅ Successfully saved locally to {local_path}.")


Attempting to push adapters to Hugging Face Hub: haitransistor/arcee-vylinh-vimedaqa-qlora-adapters


README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/haitransistor/arcee-vylinh-vimedaqa-qlora-adapters


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Successfully pushed to Hub.
